In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install rasterio geopandas matplotlib scipy -q

In [ ]:
import os
import numpy as np
import rasterio
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import pandas as pd

# ── Configuração ───────────────────────────────────────────────
BASE = "/content/drive/MyDrive/comparacao"
VOOS = ["voo1", "voo2", "voo3", "voo4"]
SOFTWARES = ["pix4d", "dronedeploy", "qgis"]
INDICES = ["ndvi", "osavi", "rdvi"]
REFERENCIA = "qgis"

resultados     = []
diag_escala    = []   # diagnóstico de min/max por raster

CORES = {
    "qgis":        ("black",      0.7),
    "pix4d":       ("steelblue",  0.5),
    "dronedeploy": ("darkorange", 0.5),
    "djiterra":    ("green",      0.5),
}

# ── Funções auxiliares ─────────────────────────────────────────
def carregar_shapefile(shp_path):
    aoi = gpd.read_file(shp_path)
    if aoi is None or aoi.empty:
        raise ValueError("Shapefile vazio ou corrompido.")
    bounds = aoi.total_bounds
    if any(np.isnan(bounds)):
        raise ValueError("Shapefile sem geometria válida (bounds = NaN).")
    # Bounds em graus decimais → EPSG:4326
    if aoi.crs is None or bounds[0] < -90:
        aoi = aoi.set_crs("EPSG:4326", allow_override=True)
    return aoi


def carregar_e_recortar(caminho_tif, aoi):
    with rasterio.open(caminho_tif) as src:
        if aoi.crs != src.crs:
            aoi = aoi.to_crs(src.crs)
        geom = aoi.geometry.values
        arr, transform = mask(src, geom, crop=True, nodata=np.nan)
        arr = arr[0].astype(np.float32)
        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan
        profile = src.profile.copy()
        profile.update(count=1, dtype="float32", nodata=np.nan,
                       transform=transform,
                       width=arr.shape[1], height=arr.shape[0])
    return arr, profile


def reprojetar_para_referencia(arr_src, profile_src, profile_ref):
    arr_out = np.full((profile_ref["height"], profile_ref["width"]),
                      np.nan, dtype=np.float32)
    reproject(
        source=arr_src, destination=arr_out,
        src_transform=profile_src["transform"], src_crs=profile_src["crs"],
        dst_transform=profile_ref["transform"], dst_crs=profile_ref["crs"],
        resampling=Resampling.bilinear,
        src_nodata=np.nan, dst_nodata=np.nan,
    )
    return arr_out


def calcular_metricas(ref, comp, nome_software, voo, indice):
    validos = ~np.isnan(ref) & ~np.isnan(comp)
    r = ref[validos].ravel()
    c = comp[validos].ravel()
    if len(r) < 10:
        print(f"    ⚠ Pixels válidos insuficientes: {nome_software} | {indice}")
        return
    rmse    = np.sqrt(np.mean((c - r) ** 2))
    mae     = np.mean(np.abs(c - r))
    r2, _   = pearsonr(r, c)
    bias    = np.mean(c - r)          # viés sistemático
    resultados.append({
        "voo": voo, "indice": indice, "software": nome_software,
        "RMSE": round(rmse, 5), "MAE": round(mae, 5),
        "R":    round(r2, 4),   "Bias": round(bias, 5),
    })


def salvar_mapa_diferenca(arr_diff, profile, caminho_saida):
    profile.update(dtype="float32", count=1, nodata=np.nan)
    os.makedirs(os.path.dirname(caminho_saida), exist_ok=True)
    with rasterio.open(caminho_saida, "w", **profile) as dst:
        dst.write(arr_diff, 1)


def registrar_escala(arr, sw, voo, indice):
    validos = arr[~np.isnan(arr)]
    if len(validos) == 0:
        return
    diag_escala.append({
        "voo": voo, "indice": indice, "software": sw,
        "min":  round(float(validos.min()), 4),
        "max":  round(float(validos.max()), 4),
        "mean": round(float(validos.mean()), 4),
        "std":  round(float(validos.std()), 4),
    })


# ── Loop principal ─────────────────────────────────────────────
for voo in VOOS:
    print(f"\n{'='*40}\n{voo.upper()}\n{'='*40}")

    shp_path = os.path.join(BASE, voo, f"mascara_{voo}.shp")
    if not os.path.exists(shp_path):
        print(f"  ⚠ Shapefile não encontrado: {shp_path}")
        continue
    try:
        aoi = carregar_shapefile(shp_path)
    except Exception as e:
        print(f"  ⚠ Erro no shapefile: {e}")
        continue

    for indice in INDICES:
        print(f"\n  Índice: {indice.upper()}")

        path_ref = os.path.join(BASE, voo, REFERENCIA, f"{indice}.tif")
        if not os.path.exists(path_ref):
            print(f"    ⚠ Referência não encontrada: {path_ref}")
            continue
        try:
            arr_ref, profile_ref = carregar_e_recortar(path_ref, aoi)
        except Exception as e:
            print(f"    ⚠ Erro ao carregar referência: {e}")
            continue

        registrar_escala(arr_ref, "qgis", voo, indice)

        # ── Coleta arrays de todos os softwares para definir xlim dinâmico ──
        arrays = {"qgis": arr_ref}
        for sw in SOFTWARES:
            if sw == REFERENCIA:
                continue
            path_sw = os.path.join(BASE, voo, sw, f"{indice}.tif")
            if not os.path.exists(path_sw):
                print(f"    ⚠ Não encontrado: {path_sw}")
                continue
            try:
                arr_sw, profile_sw = carregar_e_recortar(path_sw, aoi)
                arr_sw_r = reprojetar_para_referencia(arr_sw, profile_sw, profile_ref)
                arrays[sw] = arr_sw_r
                registrar_escala(arr_sw_r, sw, voo, indice)
            except Exception as e:
                print(f"    ⚠ Erro ao carregar {sw}: {e}")

        # ── xlim baseado no percentil 1–99 de todos os dados ──
        todos = np.concatenate([a[~np.isnan(a)].ravel() for a in arrays.values()])
        xmin  = np.percentile(todos, 1)
        xmax  = np.percentile(todos, 99)
        margem = (xmax - xmin) * 0.05
        xlim  = (xmin - margem, xmax + margem)

        # ── Histograma sobreposto ──────────────────────────────
        fig_hist, ax_hist = plt.subplots(figsize=(9, 4))

        # Plota softwares primeiro, QGIS por último (fica visível em cima)
        ordem = [sw for sw in arrays if sw != "qgis"] + ["qgis"]
        for sw in ordem:
            arr = arrays[sw]
            cor, alpha = CORES.get(sw, ("gray", 0.5))
            vals = arr[~np.isnan(arr)].ravel()
            ax_hist.hist(vals, bins=200, alpha=alpha,
                         label=f"{sw} (μ={vals.mean():.3f})",
                         color=cor, range=xlim)
            # Linha vertical na média
            ax_hist.axvline(vals.mean(), color=cor,
                            linestyle="--", linewidth=1.2)

        ax_hist.set_title(f"{voo} | {indice.upper()} — Histogramas sobrepostos")
        ax_hist.set_xlabel("Valor do índice")
        ax_hist.set_ylabel("Frequência")
        ax_hist.set_xlim(xlim)
        ax_hist.legend(fontsize=8)
        out_hist = os.path.join(BASE, "resultados", voo, f"hist_{indice}.png")
        os.makedirs(os.path.dirname(out_hist), exist_ok=True)
        fig_hist.savefig(out_hist, dpi=150, bbox_inches="tight")
        plt.close(fig_hist)
        print(f"    ✔ Histograma salvo")

        # ── Scatter + métricas ─────────────────────────────────
        for sw, arr_sw_r in arrays.items():
            if sw == "qgis":
                continue
            calcular_metricas(arr_ref, arr_sw_r, sw, voo, indice)

            validos = ~np.isnan(arr_ref) & ~np.isnan(arr_sw_r)
            r_flat  = arr_ref[validos].ravel()
            c_flat  = arr_sw_r[validos].ravel()
            n       = min(50000, len(r_flat))
            idx     = np.random.choice(len(r_flat), n, replace=False)

            cor, _ = CORES.get(sw, ("blue", 0.5))
            fig_sc, ax_sc = plt.subplots(figsize=(5, 5))
            ax_sc.scatter(r_flat[idx], c_flat[idx], s=0.5, alpha=0.3, color=cor)

            # Linha 1:1 no range real dos dados
            lim_min = min(r_flat.min(), c_flat.min())
            lim_max = max(r_flat.max(), c_flat.max())
            ax_sc.plot([lim_min, lim_max], [lim_min, lim_max],
                       "r--", linewidth=1, label="1:1")

            # Anotação de métricas no gráfico
            met = [m for m in resultados
                   if m["voo"]==voo and m["indice"]==indice and m["software"]==sw]
            if met:
                m = met[-1]
                ax_sc.annotate(
                    f"R={m['R']:.4f}\nRMSE={m['RMSE']:.5f}\nBias={m['Bias']:.5f}",
                    xy=(0.05, 0.95), xycoords="axes fraction",
                    va="top", fontsize=8,
                    bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.7)
                )

            ax_sc.set_xlabel("QGIS (referência)")
            ax_sc.set_ylabel(sw)
            ax_sc.set_title(f"{voo} | {indice.upper()} | {sw} vs QGIS")
            ax_sc.legend(fontsize=8)

            out_sc = os.path.join(BASE, "resultados", voo,
                                  f"scatter_{indice}_{sw}.png")
            fig_sc.savefig(out_sc, dpi=150, bbox_inches="tight")
            plt.close(fig_sc)
            print(f"    ✔ Scatter salvo: {sw}")

            # Mapa de diferença
            diff     = arr_sw_r - arr_ref
            out_diff = os.path.join(BASE, "resultados", voo,
                                    f"diff_{indice}_{sw}.tif")
            salvar_mapa_diferenca(diff, profile_ref.copy(), out_diff)
            print(f"    ✔ Diferença salva: {sw}")

# ── Exporta tabela de métricas ─────────────────────────────────
if resultados:
    df = pd.DataFrame(resultados)
    out_csv = os.path.join(BASE, "resultados", "metricas.csv")
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)
    df.to_csv(out_csv, index=False)
    print(f"\n✅ Métricas salvas em: {out_csv}")
    print(df.to_string(index=False))

# ── Exporta diagnóstico de escala ──────────────────────────────
if diag_escala:
    df_diag = pd.DataFrame(diag_escala)
    out_diag = os.path.join(BASE, "resultados", "diagnostico_escala.csv")
    df_diag.to_csv(out_diag, index=False)
    print(f"\n📊 Diagnóstico de escala salvo em: {out_diag}")
    print(df_diag.to_string(index=False))

if not resultados:
    print("\n⚠ Nenhuma métrica calculada. Verifique os caminhos e arquivos.")

In [ ]:
import os
import numpy as np
import rasterio
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import pandas as pd

# ── Configuração ───────────────────────────────────────────────
BASE       = "/content/drive/MyDrive/comparacao"
VOOS       = ["voo1", "voo2", "voo3", "voo4"]
SOFTWARES  = ["pix4d", "dronedeploy", "qgis"]
INDICES    = ["ndvi", "osavi", "rdvi"]
REFERENCIA = "qgis"

CORES = {
    "qgis":        ("black",      0.7),
    "pix4d":       ("steelblue",  0.5),
    "dronedeploy": ("darkorange", 0.5),
}

resultados  = []   # métricas originais e corrigidas
diag_escala = []   # min/max/mean/std por raster


# ════════════════════════════════════════════════════════════════
# FUNÇÕES BASE
# ════════════════════════════════════════════════════════════════
def carregar_shapefile(shp_path):
    aoi = gpd.read_file(shp_path)
    if aoi is None or aoi.empty:
        raise ValueError("Shapefile vazio ou corrompido.")
    bounds = aoi.total_bounds
    if any(np.isnan(bounds)):
        raise ValueError("Shapefile sem geometria válida (bounds = NaN).")
    if aoi.crs is None or bounds[0] < -90:
        aoi = aoi.set_crs("EPSG:4326", allow_override=True)
    return aoi


def carregar_e_recortar(caminho_tif, aoi):
    with rasterio.open(caminho_tif) as src:
        if aoi.crs != src.crs:
            aoi = aoi.to_crs(src.crs)
        geom = aoi.geometry.values
        arr, transform = mask(src, geom, crop=True, nodata=np.nan)
        arr = arr[0].astype(np.float32)
        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan
        profile = src.profile.copy()
        profile.update(count=1, dtype="float32", nodata=np.nan,
                       transform=transform,
                       width=arr.shape[1], height=arr.shape[0])
    return arr, profile


def reprojetar_para_referencia(arr_src, profile_src, profile_ref):
    arr_out = np.full((profile_ref["height"], profile_ref["width"]),
                      np.nan, dtype=np.float32)
    reproject(
        source=arr_src, destination=arr_out,
        src_transform=profile_src["transform"], src_crs=profile_src["crs"],
        dst_transform=profile_ref["transform"], dst_crs=profile_ref["crs"],
        resampling=Resampling.bilinear,
        src_nodata=np.nan, dst_nodata=np.nan,
    )
    return arr_out


def salvar_tif(arr, profile, caminho):
    profile.update(dtype="float32", count=1, nodata=np.nan)
    os.makedirs(os.path.dirname(caminho), exist_ok=True)
    with rasterio.open(caminho, "w", **profile) as dst:
        dst.write(arr, 1)


def registrar_escala(arr, sw, voo, indice):
    v = arr[~np.isnan(arr)]
    if len(v) == 0:
        return
    diag_escala.append({
        "voo": voo, "indice": indice, "software": sw,
        "min":  round(float(v.min()),  4),
        "max":  round(float(v.max()),  4),
        "mean": round(float(v.mean()), 4),
        "std":  round(float(v.std()),  4),
    })


def calcular_e_guardar(ref, comp, sw, voo, indice, metodo="original"):
    validos = ~np.isnan(ref) & ~np.isnan(comp)
    r = ref[validos].ravel()
    c = comp[validos].ravel()
    if len(r) < 10:
        return None
    rmse   = np.sqrt(np.mean((c - r) ** 2))
    mae    = np.mean(np.abs(c - r))
    rval,_ = pearsonr(r, c)
    bias   = np.mean(c - r)
    entrada = {
        "voo": voo, "indice": indice, "software": sw, "metodo": metodo,
        "RMSE":  round(rmse,  5),
        "MAE":   round(mae,   5),
        "R":     round(rval,  4),
        "Bias":  round(bias,  5),
    }
    resultados.append(entrada)
    return entrada


# ════════════════════════════════════════════════════════════════
# MÉTODOS DE CORREÇÃO DE ESCALA (aplicados pixel a pixel)
# ════════════════════════════════════════════════════════════════
def corr_formula_ndvi(arr):
    """
    Tenta converter NIR/(NIR+Red) → NDVI padrão.
    NIR/(NIR+Red) = x  →  NDVI = 2x - 1
    Válido apenas se x ∈ [0, 1].
    """
    arr_c = arr.copy()
    mask_valido = (arr >= 0) & (arr <= 1)
    arr_c[mask_valido]  = arr[mask_valido] * 2 - 1
    arr_c[~mask_valido] = np.nan
    return arr_c


def corr_zscore(arr, ref):
    """
    Padroniza arr para ter mesma média e desvio da referência
    (comparação de padrão espacial, independente de escala absoluta).
    """
    v_arr = arr[~np.isnan(arr)]
    v_ref = ref[~np.isnan(ref)]
    if v_arr.std() == 0:
        return arr.copy()
    return (arr - v_arr.mean()) / v_arr.std() * v_ref.std() + v_ref.mean()


def corr_minmax(arr, ref):
    """
    Estica lineamente o range de arr para o range da referência
    usando percentis 2–98 para robustez a outliers.
    """
    v_arr = arr[~np.isnan(arr)]
    v_ref = ref[~np.isnan(ref)]
    a_min, a_max = np.percentile(v_arr, [2, 98])
    r_min, r_max = np.percentile(v_ref, [2, 98])
    if a_max == a_min:
        return arr.copy()
    arr_c = (arr - a_min) / (a_max - a_min) * (r_max - r_min) + r_min
    return arr_c


def corr_offset(arr, ref):
    """
    Remove apenas o bias médio, mantendo a variância original.
    """
    v_arr = arr[~np.isnan(arr)]
    v_ref = ref[~np.isnan(ref)]
    bias  = v_arr.mean() - v_ref.mean()
    return arr - bias


# ════════════════════════════════════════════════════════════════
# PLOTAGEM
# ════════════════════════════════════════════════════════════════
def plot_histogramas(arrays_orig, arrays_corr, voo, indice, pasta):
    """
    Dois painéis: histogramas originais (esq.) e após melhor correção (dir.).
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    for ax, arrs, titulo in zip(
        axes,
        [arrays_orig, arrays_corr],
        ["Original", "Após correção (z-score)"]
    ):
        todos = np.concatenate([a[~np.isnan(a)].ravel() for a in arrs.values()])
        xmin, xmax = np.percentile(todos, 1), np.percentile(todos, 99)
        margem = (xmax - xmin) * 0.05
        xlim = (xmin - margem, xmax + margem)

        ordem = [sw for sw in arrs if sw != "qgis"] + ["qgis"]
        for sw in ordem:
            arr  = arrs[sw]
            cor, alpha = CORES.get(sw, ("gray", 0.5))
            vals = arr[~np.isnan(arr)].ravel()
            ax.hist(vals, bins=200, alpha=alpha,
                    label=f"{sw} (μ={vals.mean():.3f})",
                    color=cor, range=xlim)
            ax.axvline(vals.mean(), color=cor, linestyle="--", linewidth=1.2)

        ax.set_title(f"{voo} | {indice.upper()} — {titulo}")
        ax.set_xlabel("Valor do índice")
        ax.set_ylabel("Frequência")
        ax.set_xlim(xlim)
        ax.legend(fontsize=8)

    fig.tight_layout()
    out = os.path.join(pasta, f"hist_{indice}.png")
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out


def plot_scatter(ref, comp, comp_corr, sw, voo, indice, pasta):
    """
    Dois painéis: scatter original (esq.) e corrigido (dir.).
    """
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))

    for ax, arr_c, titulo, met_label in zip(
        axes,
        [comp,      comp_corr],
        ["Original", "Z-score corrigido"],
        ["original", "zscore"]
    ):
        validos = ~np.isnan(ref) & ~np.isnan(arr_c)
        r_f = ref[validos].ravel()
        c_f = arr_c[validos].ravel()
        n   = min(50000, len(r_f))
        idx = np.random.choice(len(r_f), n, replace=False)

        cor, _ = CORES.get(sw, ("blue", 0.5))
        ax.scatter(r_f[idx], c_f[idx], s=0.5, alpha=0.3, color=cor)

        lim_min = min(r_f.min(), c_f.min())
        lim_max = max(r_f.max(), c_f.max())
        ax.plot([lim_min, lim_max], [lim_min, lim_max],
                "r--", linewidth=1, label="1:1")

        # Métricas já calculadas
        met = [m for m in resultados
               if m["voo"]==voo and m["indice"]==indice
               and m["software"]==sw and m["metodo"]==met_label]
        if met:
            m = met[-1]
            ax.annotate(
                f"R={m['R']:.4f}\nRMSE={m['RMSE']:.5f}\nBias={m['Bias']:.5f}",
                xy=(0.05, 0.95), xycoords="axes fraction",
                va="top", fontsize=8,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8)
            )

        ax.set_xlabel("QGIS (referência)")
        ax.set_ylabel(sw)
        ax.set_title(f"{voo} | {indice.upper()} | {sw} — {titulo}")
        ax.legend(fontsize=8)

    fig.tight_layout()
    out = os.path.join(pasta, f"scatter_{indice}_{sw}.png")
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out


# ════════════════════════════════════════════════════════════════
# LOOP PRINCIPAL
# ════════════════════════════════════════════════════════════════
for voo in VOOS:
    print(f"\n{'='*40}\n{voo.upper()}\n{'='*40}")

    shp_path = os.path.join(BASE, voo, f"mascara_{voo}.shp")
    if not os.path.exists(shp_path):
        print(f"  ⚠ Shapefile não encontrado: {shp_path}")
        continue
    try:
        aoi = carregar_shapefile(shp_path)
    except Exception as e:
        print(f"  ⚠ Erro no shapefile: {e}")
        continue

    for indice in INDICES:
        print(f"\n  Índice: {indice.upper()}")
        pasta_out = os.path.join(BASE, "resultados", voo)
        os.makedirs(pasta_out, exist_ok=True)

        # ── Referência ──────────────────────────────────────────
        path_ref = os.path.join(BASE, voo, REFERENCIA, f"{indice}.tif")
        if not os.path.exists(path_ref):
            print(f"    ⚠ Referência não encontrada: {path_ref}")
            continue
        try:
            arr_ref, profile_ref = carregar_e_recortar(path_ref, aoi)
        except Exception as e:
            print(f"    ⚠ Erro ao carregar referência: {e}")
            continue

        registrar_escala(arr_ref, "qgis", voo, indice)

        arrays_orig = {"qgis": arr_ref}
        arrays_corr = {"qgis": arr_ref}

        # ── Softwares ───────────────────────────────────────────
        for sw in SOFTWARES:
            if sw == REFERENCIA:
                continue

            path_sw = os.path.join(BASE, voo, sw, f"{indice}.tif")
            if not os.path.exists(path_sw):
                print(f"    ⚠ Não encontrado: {path_sw}")
                continue

            try:
                arr_sw, profile_sw = carregar_e_recortar(path_sw, aoi)
                arr_sw = reprojetar_para_referencia(arr_sw, profile_sw, profile_ref)
            except Exception as e:
                print(f"    ⚠ Erro ao carregar {sw}: {e}")
                continue

            registrar_escala(arr_sw, sw, voo, indice)
            arrays_orig[sw] = arr_sw

            # ── Métricas originais ───────────────────────────────
            calcular_e_guardar(arr_ref, arr_sw, sw, voo, indice, "original")

            # ── Correções (apenas DroneDeploy) ──────────────────
            if sw == "dronedeploy":
                print(f"    → Aplicando correções ao {sw}...")

                # 1. Fórmula NIR/(NIR+Red) → NDVI padrão (só para NDVI)
                if indice == "ndvi":
                    arr_form = corr_formula_ndvi(arr_sw)
                    calcular_e_guardar(arr_ref, arr_form, sw, voo, indice, "formula_ndvi")
                    salvar_tif(arr_form, profile_ref.copy(),
                               os.path.join(pasta_out, f"dd_formula_{indice}.tif"))

                # 2. Z-score (match de média e desvio)
                arr_zsc = corr_zscore(arr_sw, arr_ref)
                calcular_e_guardar(arr_ref, arr_zsc, sw, voo, indice, "zscore")
                salvar_tif(arr_zsc, profile_ref.copy(),
                           os.path.join(pasta_out, f"dd_zscore_{indice}.tif"))
                arrays_corr[sw] = arr_zsc   # usado no histograma corrigido

                # 3. Min-max stretch
                arr_mm = corr_minmax(arr_sw, arr_ref)
                calcular_e_guardar(arr_ref, arr_mm, sw, voo, indice, "minmax")
                salvar_tif(arr_mm, profile_ref.copy(),
                           os.path.join(pasta_out, f"dd_minmax_{indice}.tif"))

                # 4. Offset (remove bias médio)
                arr_off = corr_offset(arr_sw, arr_ref)
                calcular_e_guardar(arr_ref, arr_off, sw, voo, indice, "offset")
                salvar_tif(arr_off, profile_ref.copy(),
                           os.path.join(pasta_out, f"dd_offset_{indice}.tif"))

                # Mapa de diferença (original e z-score)
                salvar_tif(arr_sw  - arr_ref, profile_ref.copy(),
                           os.path.join(pasta_out, f"diff_{indice}_{sw}_original.tif"))
                salvar_tif(arr_zsc - arr_ref, profile_ref.copy(),
                           os.path.join(pasta_out, f"diff_{indice}_{sw}_zscore.tif"))

                # Scatter com painéis original vs z-score
                plot_scatter(arr_ref, arr_sw, arr_zsc, sw, voo, indice, pasta_out)
                print(f"    ✔ Scatter (original + zscore) salvo")

            else:
                # Pix4D — apenas original
                salvar_tif(arr_sw - arr_ref, profile_ref.copy(),
                           os.path.join(pasta_out, f"diff_{indice}_{sw}.tif"))
                # Scatter simples
                validos = ~np.isnan(arr_ref) & ~np.isnan(arr_sw)
                r_f = arr_ref[validos].ravel()
                c_f = arr_sw[validos].ravel()
                n   = min(50000, len(r_f))
                idx = np.random.choice(len(r_f), n, replace=False)
                cor, _ = CORES.get(sw, ("blue", 0.5))
                fig_sc, ax_sc = plt.subplots(figsize=(5, 5))
                ax_sc.scatter(r_f[idx], c_f[idx], s=0.5, alpha=0.3, color=cor)
                ax_sc.plot([r_f.min(), r_f.max()], [r_f.min(), r_f.max()],
                           "r--", linewidth=1)
                met = [m for m in resultados
                       if m["voo"]==voo and m["indice"]==indice
                       and m["software"]==sw and m["metodo"]=="original"]
                if met:
                    m = met[-1]
                    ax_sc.annotate(
                        f"R={m['R']:.4f}\nRMSE={m['RMSE']:.5f}\nBias={m['Bias']:.5f}",
                        xy=(0.05, 0.95), xycoords="axes fraction",
                        va="top", fontsize=8,
                        bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8)
                    )
                ax_sc.set_xlabel("QGIS (referência)")
                ax_sc.set_ylabel(sw)
                ax_sc.set_title(f"{voo} | {indice.upper()} | {sw} vs QGIS")
                out_sc = os.path.join(pasta_out, f"scatter_{indice}_{sw}.png")
                fig_sc.savefig(out_sc, dpi=150, bbox_inches="tight")
                plt.close(fig_sc)
                print(f"    ✔ Scatter {sw} salvo")
                arrays_corr[sw] = arr_sw

        # ── Histogramas sobrepostos (original + corrigido) ──────
        if len(arrays_orig) > 1:
            plot_histogramas(arrays_orig, arrays_corr, voo, indice, pasta_out)
            print(f"    ✔ Histogramas salvos")


# ════════════════════════════════════════════════════════════════
# EXPORTA TABELAS
# ════════════════════════════════════════════════════════════════
pasta_res = os.path.join(BASE, "resultados")
os.makedirs(pasta_res, exist_ok=True)

# Métricas completas
if resultados:
    df = pd.DataFrame(resultados)

    # Tabela completa
    df.to_csv(os.path.join(pasta_res, "metricas_completas.csv"), index=False)

    # Tabela resumo: melhor método por software/voo/índice (maior R)
    df_dd   = df[df["software"] == "dronedeploy"].copy()
    df_rest = df[df["software"] != "dronedeploy"].copy()
    df_best_dd = (df_dd.loc[df_dd.groupby(["voo","indice","software"])["R"].idxmax()]
                  .reset_index(drop=True))
    df_resumo = pd.concat([df_rest, df_best_dd], ignore_index=True)
    df_resumo = df_resumo.sort_values(["voo","indice","software"]).reset_index(drop=True)
    df_resumo.to_csv(os.path.join(pasta_res, "metricas_resumo.csv"), index=False)

    print(f"\n✅ metricas_completas.csv — todas as correções")
    print(f"✅ metricas_resumo.csv    — melhor método por software\n")
    print(df_resumo.to_string(index=False))

# Diagnóstico de escala
if diag_escala:
    pd.DataFrame(diag_escala).to_csv(
        os.path.join(pasta_res, "diagnostico_escala.csv"), index=False)
    print(f"\n📊 diagnostico_escala.csv salvo")

if not resultados:
    print("\n⚠ Nenhuma métrica calculada. Verifique caminhos e arquivos.")